In [37]:
import pandas as pd
df = pd.read_csv('rawdata.csv')
print("DataSet Shape : ", df.shape)
display(df.head())

DataSet Shape :  (9215, 14)


,Order ID,Order Date,Origin Port,Carrier,TPT,Service Level,Ship ahead day count,Ship Late Day count,Customer,Product ID,Plant Code,Destination Port,Unit quantity,Weight
0,1447296447,26-05-2013,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,808,14.30
1,1447158015,26-05-2013,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,3188,87.94
2,1447138899,26-05-2013,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,2331,61.20
3,1447363528,26-05-2013,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,847,16.16
4,1447363981,26-05-2013,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,2163,52.34


In [38]:
print("--Data Info--")
df.info()

print("\n--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Duplicate Rows ---")
print("Total duplicates:", df.duplicated().sum())

--Data Info--
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9215 entries, 0 to 9214
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Order ID              9215 non-null   int64  
 1   Order Date            9215 non-null   object 
 2   Origin Port           9215 non-null   object 
 3   Carrier               9215 non-null   object 
 4   TPT                   9215 non-null   int64  
 5   Service Level         9215 non-null   object 
 6   Ship ahead day count  9215 non-null   int64  
 7   Ship Late Day count   9215 non-null   int64  
 8   Customer              9215 non-null   object 
 9   Product ID            9215 non-null   int64  
 10  Plant Code            9215 non-null   object 
 11  Destination Port      9215 non-null   object 
 12  Unit quantity         9215 non-null   int64  
 13  Weight                9215 non-null   float64
dtypes: float64(1), int64(6), object(7)
memory usage: 1008.0+ K

In [39]:
import numpy as np

# Create a copy of the dataframe to keep the original safe
df_dirty = df.copy()

# 1. Introduce 5% missing values into the 'Carrier' column
carrier_mask = np.random.rand(len(df_dirty)) < 0.05
df_dirty.loc[carrier_mask, 'Carrier'] = np.nan

# 2. Introduce 10% missing values into the 'Service Level' column
service_mask = np.random.rand(len(df_dirty)) < 0.10
df_dirty.loc[service_mask, 'Service Level'] = np.nan

# 3. Simulate a common sensor/system error: replacing 0s in 'Weight' with NaN
df_dirty.loc[df_dirty['Weight'] == 0, 'Weight'] = np.nan

# Verify the newly created missing values
print("--- Simulated Missing Values ---")
print(df_dirty[['Carrier', 'Service Level', 'Weight']].isnull().sum())

--- Simulated Missing Values ---
Carrier          457
Service Level    896
Weight             2
dtype: int64


In [40]:
# 1. Fix Categories: Fill missing text with the most frequent item (Mode)
df_dirty['Carrier'] = df_dirty['Carrier'].fillna(df_dirty['Carrier'].mode()[0])
df_dirty['Service Level'] = df_dirty['Service Level'].fillna(df_dirty['Service Level'].mode()[0])

# 2. Fix Numbers: Fill missing weights with the middle value (Median)
df_dirty['Weight'] = df_dirty['Weight'].fillna(df_dirty['Weight'].median())

# 3. Verify the fix
print("--- Remaining Missing Values ---")
print(df_dirty[['Carrier', 'Service Level', 'Weight']].isnull().sum())

--- Remaining Missing Values ---
Carrier          0
Service Level    0
Weight           0
dtype: int64


In [41]:
df_dirty['Order Date'] = pd.to_datetime(df_dirty['Order Date'], dayfirst=True, errors='coerce')

df_dirty['Order Month'] = df_dirty['Order Date'].dt.month
df_dirty['Order DayOfWeek'] = df_dirty['Order Date'].dt.dayofweek

df_dirty = df_dirty.drop(columns=['Order Date', 'Order ID'])

print("--- Engineered Features ---")
display(df_dirty[['Order Month', 'Order DayOfWeek']].head())
print("\nCurrent Columns:", df_dirty.columns.tolist())

--- Engineered Features ---


,Order Month,Order DayOfWeek
0,5,6
1,5,6
2,5,6
3,5,6
4,5,6



Current Columns: ['Origin Port', 'Carrier', 'TPT', 'Service Level', 'Ship ahead day count', 'Ship Late Day count', 'Customer', 'Product ID', 'Plant Code', 'Destination Port', 'Unit quantity', 'Weight', 'Order Month', 'Order DayOfWeek']


In [42]:
categorical_cols = ['Origin Port', 'Carrier', 'Service Level', 'Customer', 'Plant Code', 'Destination Port']

df_encoded = pd.get_dummies(df_dirty, columns=categorical_cols, drop_first=True)
print("Old shape:", df_dirty.shape)
print("New shape:", df_encoded.shape)
display(df_encoded.head())

Old shape: (9215, 14)
New shape: (9215, 65)


,TPT,Ship ahead day count,Ship Late Day count,Product ID,Unit quantity,Weight,Order Month,Order DayOfWeek,Origin Port_PORT05,Origin Port_PORT09,...,Customer_V55_13,Customer_V55_37,Customer_V55_39,Customer_V55_47,Plant Code_PLANT04,Plant Code_PLANT08,Plant Code_PLANT09,Plant Code_PLANT12,Plant Code_PLANT13,Plant Code_PLANT16
0,1,3,0,1700106,808,14.30,5,6,False,True,...,False,False,False,False,False,False,False,False,False,True
1,1,3,0,1700106,3188,87.94,5,6,False,True,...,False,False,False,False,False,False,False,False,False,True
2,1,3,0,1700106,2331,61.20,5,6,False,True,...,False,False,False,False,False,False,False,False,False,True
3,1,3,0,1700106,847,16.16,5,6,False,True,...,False,False,False,False,False,False,False,False,False,True
4,1,3,0,1700106,2163,52.34,5,6,False,True,...,False,False,False,False,False,False,False,False,False,True


In [43]:
%pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [44]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns=['Ship Late Day count'])
y = df_encoded['Ship Late Day count']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training Data Shape:", X_train_scaled.shape)
print("Testing Data Shape:", X_test_scaled.shape)

Training Data Shape: (7372, 64)
Testing Data Shape: (1843, 64)


In [45]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
print("Training Random Forest model...")
rf_model.fit(X_train_scaled, y_train)

predictions = rf_model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"Mean Absolute Error: {mae:.2f} days")
print(f"R-Squared Score: {r2:.2f}")

Training Random Forest model...
Mean Absolute Error: 0.02 days
R-Squared Score: 0.67


In [46]:
import joblib


joblib.dump(rf_model, 'rf_inventory_model.pkl')
joblib.dump(scaler, 'scaler.pkl')


joblib.dump(X.columns.tolist(), 'model_columns.pkl')

print("Artifacts successfully saved to disk.")

Artifacts successfully saved to disk.


In [47]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

lr_predictions = lr_model.predict(X_test_scaled)

lr_mae = mean_absolute_error(y_test, lr_predictions)
lr_r2 = r2_score(y_test, lr_predictions)

print("=== Model Performance Comparison ===")
print(f"Linear Regression  -> MAE: {lr_mae:.4f} days | R2 Score: {lr_r2:.4f}")
print(f"Random Forest      -> MAE: {mae:.4f} days | R2 Score: {r2:.4f}")

=== Model Performance Comparison ===
Linear Regression  -> MAE: 0.0640 days | R2 Score: 0.2795
Random Forest      -> MAE: 0.0248 days | R2 Score: 0.6714
